# Economic Calendar API

**TradingView Economic Calendar** — Türkiye filtresi.

Belirli tarih aralığında Türkiye için açıklanmış / açıklanacak makro ekonomik olayları listeler (faiz kararı, enflasyon, işsizlik, vb.).

## Endpoint

```
GET https://economic-calendar.tradingview.com/events?from={from}&to={to}&countries=TR
```

**Header'lar:** `accept: application/json` · `origin: https://www.tradingview.com` · `referer: https://www.tradingview.com/` · `user-agent: <browser UA>` (zorunlu)

`user-agent` boş veya bot benzeri ise istek reddedilebilir.

## İstek (Request)

### Query Parametreleri

| Alan | Açıklama |
|------|----------|
| `from` | Başlangıç zamanı · ISO 8601 (UTC) — örn. `2026-04-19T21:00:00.000Z` |
| `to` | Bitiş zamanı · ISO 8601 (UTC) — örn. `2026-04-26T21:00:00.000Z` |
| `countries` | Ülke kodu filtresi · Türkiye için `TR` |

### Headers

| Header | Değer |
|--------|-------|
| `accept` | `application/json` |
| `origin` | `https://www.tradingview.com` |
| `referer` | `https://www.tradingview.com/` |
| `user-agent` | Browser user agent · **zorunlu** |

## Yanıt (Response)

`{ status, result[] }` → `result`, ekonomik olayların dizisi.

### `event`

| Alan | Açıklama |
|------|----------|
| `id` | Benzersiz event ID |
| `title` | Event başlığı (örn. `TCMB Interest Rate Decision`) |
| `country` | Ülke kodu · örn. `TR` |
| `indicator` | Makro gösterge adı |
| `ticker` | TradingView ekonomi sembolü |
| `actual` / `previous` / `forecast` | Açıklanan, önceki ve beklenti değerleri |
| `unit` / `scale` | `%`, `TRY`, `$`, `B`, `T` gibi birim ve ölçek |
| `importance` | Etki seviyesi · `-1` düşük, `0` orta, `1` yüksek |
| `date` | Yayın tarihi-saati · ISO 8601 (UTC) |

## 1. Ortak Ayarlar

In [1]:
import requests
from datetime import datetime, timedelta, timezone

URL = "https://economic-calendar.tradingview.com/events"

HEADERS = {
    "accept": "application/json",
    "accept-language": "en-US,en;q=0.9,tr;q=0.8",
    "origin": "https://www.tradingview.com",
    "referer": "https://www.tradingview.com/",
    "user-agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    ),
}


def iso(dt):
    """datetime → ISO 8601 UTC (millisecond precision, Z son ekiyle)."""
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

## 2. Tarama Fonksiyonu

Verilen tarih aralığı ve ülke kodu için olayları çeker.

In [2]:
def takvim(start, end, countries="TR"):
    params = {
        "from": iso(start),
        "to": iso(end),
        "countries": countries,
    }
    r = requests.get(URL, headers=HEADERS, params=params)
    r.raise_for_status()
    return r.json()


def yazdir(js, baslik):
    events = js.get("result", [])
    print(f"{baslik} · status={js.get('status')} · {len(events)} olay\n")
    print(f"{'Tarih (UTC)':<20} {'Önem':>4}  {'Önceki':>10} {'Beklenti':>10} {'Açıklanan':>10}  Başlık")
    print("-" * 100)
    for e in events:
        d = e.get("date", "")[:16].replace("T", " ")
        imp = {1: "YÜK", 0: "ORT", -1: "DÜŞ"}.get(e.get("importance"), "-")
        unit = e.get("unit") or ""
        prev = f"{e.get('previous', '')}{unit}" if e.get("previous") is not None else "-"
        fcst = f"{e.get('forecast', '')}{unit}" if e.get("forecast") is not None else "-"
        actual = f"{e.get('actual', '')}{unit}" if e.get("actual") is not None else "-"
        print(f"{d:<20} {imp:>4}  {prev:>10} {fcst:>10} {actual:>10}  {e.get('title', '')}")

## 3. Bu Hafta · Türkiye Olayları

Bugünden itibaren ileri 7 günlük pencere.

In [3]:
now = datetime.now(timezone.utc)
js = takvim(now, now + timedelta(days=7))
yazdir(js, "Türkiye · önümüzdeki 7 gün")

Türkiye · önümüzdeki 7 gün · status=ok · 19 olay

Tarih (UTC)          Önem      Önceki   Beklenti  Açıklanan  Başlık
----------------------------------------------------------------------------------------------------
2026-04-29 07:00      DÜŞ        97.9          -          -  Economic Confidence Index
2026-04-29 07:00      ORT         8.5          -          -  Unemployment Rate
2026-04-29 07:00      DÜŞ       52.6%          -          -  Participation Rate
2026-04-30 07:00      ORT         -9$          -          -  Balance of Trade Final
2026-04-30 07:00      DÜŞ        15.2          -          -  Tourism Revenues
2026-04-30 07:00      DÜŞ         30$          -          -  Imports Final
2026-04-30 07:00      DÜŞ         21$          -          -  Exports Final
2026-04-30 08:00      DÜŞ       -2.08          -          -  Tourist Arrivals YoY
2026-04-30 11:00      DÜŞ           -          -          -  MPC Meeting Summary
2026-04-30 11:30      DÜŞ       61.82          -          - 

## 4. Geçen Hafta · Açıklanmış Olaylar

Geriye dönük 7 gün — `actual` doluysa açıklama gerçekleşmiştir.

In [4]:
js = takvim(now - timedelta(days=7), now)
yazdir(js, "Türkiye · geçen 7 gün")

Türkiye · geçen 7 gün · status=ok · 6 olay

Tarih (UTC)          Önem      Önceki   Beklenti  Açıklanan  Başlık
----------------------------------------------------------------------------------------------------
2026-04-22 07:00      ORT          85          -       85.5  Consumer Confidence
2026-04-22 11:00      ORT         37%          -        37%  TCMB Interest Rate Decision
2026-04-22 11:00      DÜŞ         40%        40%        40%  Overnight Lending Rate
2026-04-22 11:00      DÜŞ       35.5%      35.5%      35.5%  Overnight Borrowing Rate
2026-04-23 00:00      DÜŞ           -          -          -  National Sovereignty and Children’s Day
2026-04-24 11:30      DÜŞ      64.07$          -     61.82$  Foreign Exchange Reserves


## 5. Yüksek Önemli Olaylar · Filtreleme

API tarafında `importance` filtresi yok — yanıt üzerinden filtrelenir.

In [5]:
js = takvim(now - timedelta(days=14), now + timedelta(days=14))
yuksek = [e for e in js.get("result", []) if e.get("importance") == 1]

print(f"Türkiye · ±14 gün · yüksek önemli: {len(yuksek)} olay\n")
for e in yuksek:
    d = e.get("date", "")[:16].replace("T", " ")
    print(f"{d}  {e.get('title', '')}  ({e.get('indicator', '-')})")

Türkiye · ±14 gün · yüksek önemli: 0 olay



## 6. curl Eşdeğeri · Türkiye filtresi · 1 haftalık aralık

In [ ]:
%%bash
curl -sS 'https://economic-calendar.tradingview.com/events?from=2026-04-19T21%3A00%3A00.000Z&to=2026-04-26T21%3A00%3A00.000Z&countries=TR' \
  -H 'accept: application/json' \
  -H 'accept-language: en-US,en;q=0.9,tr;q=0.8' \
  -H 'origin: https://www.tradingview.com' \
  -H 'referer: https://www.tradingview.com/' \
  -H 'user-agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36' \
  --compressed | python3 -m json.tool | head -25

## Notlar

- `from` ve `to` ISO 8601 UTC formatında zorunlu — yerel saat veya yalnız tarih kabul edilmez.
- Birden fazla ülke için `countries=TR,US,DE` gibi virgülle ayır.
- `actual` / `previous` / `forecast` `null` olabilir — sayısal hesaplamadan önce kontrol et.
- `importance`: `1` yüksek (faiz kararı, enflasyon), `0` orta, `-1` düşük.
- `user-agent` boş veya bot benzeri ise 403 dönebilir — gerçek bir browser UA gönder.